In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [4]:
df = pd.read_csv('insurance.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         100 non-null    int64  
 1   weight                      100 non-null    float64
 2   height                      100 non-null    float64
 3   income_lpa                  100 non-null    float64
 4   smoker                      100 non-null    bool   
 5   city                        100 non-null    object 
 6   occupation                  99 non-null     object 
 7   insurance_premium_category  59 non-null     object 
dtypes: bool(1), float64(3), int64(1), object(3)
memory usage: 5.7+ KB


In [6]:
df.shape

(100, 8)

In [7]:
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur$retired,High,NaN
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore$retired,High,NaN


In [8]:
df.tail()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
95,36,52.8,1.57,19.64000,False,Indore,business_owner,Low
96,26,113.8,1.54,34.01000,False,Delhi,private_job,Low
97,52,60.8,1.80,44.86000,False,Delhi,private_job,Low
98,27,101.1,1.82,28.30000,False,Kolkata,business_owner,Low
99,40,70.0,1.59,28.16664,True,Bangalore,government_job,Low


In [9]:
df.sample(10)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
87,30,82.0,1.60,25.59837,False,Hyderabad,government_job,Low
29,60,117.8,1.66,50.00000,True,Lucknow,business_owner,HIGH
53,41,101.3,1.85,30.00000,True,Delhi,government_job,MEDIUM
30,35,89.6,1.73,32.97000,False,Delhi,business_owner,Low
67,22,56.4,1.82,2.76000,False,Jaipur,student,Low
17,65,90.1,1.70,2.23000,False,Delhi$retired,MEDIUM,NaN
40,44,57.0,1.53,40.19000,True,Pune.unemployed,MEDIUM,NaN
46,42,83.0,1.57,25.57000,True,Kolkata.unemployed,High,NaN
74,63,71.1,1.54,1.88000,True,Jalandhar$retired,High,NaN
19,24,111.2,1.60,2.79000,True,Lucknow,student,High


In [10]:
df.isnull().sum()

age                            0
weight                         0
height                         0
income_lpa                     0
smoker                         0
city                           0
occupation                     1
insurance_premium_category    41
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df['occupation'].unique()

array(['High', 'freelancer', 'student', 'government_job', nan,
       'business_owner', 'MEDIUM', 'private_job', 'Low'], dtype=object)

In [13]:
df_feat = df.copy()

In [14]:
df_feat.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur$retired,High,NaN
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore$retired,High,NaN


In [15]:
# Feature 1: BMI (Body Mass Index)
df_feat['bmi'] = df_feat['weight'] / (df_feat['height'] ** 2)

In [16]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [17]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

In [18]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [19]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1)

In [20]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [21]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [22]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [23]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].head()

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,2.92,High,49.227482,senior,medium,3,NaN
1,34.28,freelancer,30.189017,adult,medium,1,Low
2,36.64,freelancer,21.118382,adult,low,2,Low
3,3.34,student,45.535900,young,high,1,Medium
4,3.94,High,24.296875,senior,medium,3,NaN


In [ ]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

# Drop rows with missing values in features or target
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

In [25]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,3,2.92000,High
1,30.189017,adult,medium,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,medium,3,3.94000,High
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,medium,1,34.01000,private_job
97,18.765432,middle_aged,low,1,44.86000,private_job
98,30.521676,adult,medium,1,28.30000,business_owner


In [26]:
y

0        NaN
1        Low
2        Low
3     Medium
4        NaN
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: object

In [27]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [28]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [29]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

ValueError: Input contains NaN